# ARC-AGI-3 recursive-transformer agent

This notebook uses the supplied ARC-AGI-3 environment as its input: the agent receives `FrameData` history and returns a `GameAction`. In competition reruns, `main.py` runs the agent against the gateway. Outside reruns, the notebook writes the scaffold's example `submission.parquet` (`row_id`, `game_id`, `end_of_game`, `score`) and the generated `my_agent.py`.

The recursive transformer below is an action-policy architecture adapted from the repository's RTA pattern. ARC-AGI-3 does not provide labeled action demonstrations in this notebook input, so it cannot train itself here. For meaningful transformer actions, attach a compatible trained checkpoint; without one, the agent explicitly falls back to the original random baseline. This is not a claim that a randomly initialized network solves the games.

In [ ]:
# Install from the competition's offline wheels.
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/my_agent.py
import glob, os, random, time
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent

class RecursiveActionTransformer(nn.Module):
    """Recursive [observation | draft y | latent z] encoder with action/coordinate heads."""
    def __init__(self, n_actions, d_model=128, heads=4, layers=2, max_tokens=1024, z_slots=8):
        super().__init__()
        self.max_tokens, self.z_slots = max_tokens, z_slots
        self.token = nn.Embedding(17, d_model)  # colors 0-15 plus unknown/pad token 16
        self.position = nn.Embedding(max_tokens, d_model)
        self.token_type = nn.Embedding(3, d_model)  # observation, draft, latent
        block = nn.TransformerEncoderLayer(d_model, heads, 4*d_model, dropout=0.0, batch_first=True)
        self.core = nn.TransformerEncoder(block, layers, enable_nested_tensor=False)
        self.initial_y = nn.Parameter(torch.zeros(1, 1, d_model))
        self.initial_z = nn.Parameter(torch.randn(1, z_slots, d_model)*0.02)
        self.update_y = nn.Linear(d_model, d_model)
        self.update_z = nn.Linear(d_model, d_model)
        self.action_head = nn.Linear(d_model, n_actions)
        self.xy_head = nn.Linear(d_model, 128)  # independent 64-way x and y coordinates
    def forward(self, tokens, n_sup=4, n_inner=2):
        b, s = tokens.shape
        if s > self.max_tokens: tokens = tokens[:, :self.max_tokens]; s = self.max_tokens
        pos = self.position(torch.arange(s, device=tokens.device))[None]
        x = self.token(tokens) + pos + self.token_type.weight[0]
        y = self.initial_y.expand(b, -1, -1) + self.token_type.weight[1]
        z = self.initial_z.expand(b, -1, -1) + self.token_type.weight[2]
        for _ in range(n_sup):
            for _ in range(n_inner):
                out = self.core(torch.cat((x, y, z), dim=1))
                z = z + self.update_z(out[:, s+1:, :])
            out = self.core(torch.cat((x, y, z), dim=1))
            y = y + self.update_y(out[:, s:s+1, :])
        pooled = y[:, 0]
        return self.action_head(pooled), self.xy_head(pooled).reshape(b, 2, 64)

def _as_grid(value):
    try:
        a = np.asarray(value)
        if a.ndim == 2 and a.size and a.dtype.kind in 'biuf' and np.isfinite(a).all() and a.min() >= 0 and a.max() <= 15:
            return a.astype(np.int64)
    except (TypeError, ValueError):
        pass
    return None

def _observation_tokens(frame, limit):
    # FrameData schemas vary by engine version; inspect common fields and mappings.
    queue = [frame]; seen = set(); examined = 0
    while queue and examined < 256:
        obj = queue.pop(0); examined += 1
        if id(obj) in seen: continue
        seen.add(id(obj))
        if isinstance(obj, dict):
            for key in ('grid','observation','board','pixels','data','frame'):
                if key in obj:
                    g = _as_grid(obj[key])
                    if g is not None: return np.clip(g.reshape(-1)[:limit], 0, 15)
                    queue.append(obj[key])
        else:
            for key in ('grid','observation','board','pixels','data','frame'):
                try: value = getattr(obj, key)
                except Exception: continue
                if value is not None:
                    g = _as_grid(value)
                    if g is not None: return np.clip(g.reshape(-1)[:limit], 0, 15)
                    if isinstance(value, (dict,list,tuple,np.ndarray)): queue.append(value)
    return np.array([16], dtype=np.int64)

class MyAgent(Agent):
    MAX_ACTIONS = float('inf')
    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self.rng = random.Random(time.time_ns() ^ (hash(self.game_id) & 0xffffffff))
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.actions = [a for a in GameAction if a is not GameAction.RESET]
        self.policy = None
        checkpoint = os.getenv('ARC_TRANSFORMER_CHECKPOINT')
        if not checkpoint:
            files = glob.glob('/kaggle/input/**/*.pt', recursive=True) + glob.glob('/kaggle/input/**/*.pth', recursive=True)
            checkpoint = files[0] if files else None
        if checkpoint and os.path.isfile(checkpoint):
            try:
                saved = torch.load(checkpoint, map_location=self.device, weights_only=False)
                cfg = saved.get('config', {})
                action_values = [a.value for a in self.actions]
                if saved.get('actions', action_values) != action_values:
                    raise ValueError('checkpoint action ordering does not match this arcengine version')
                self.policy = RecursiveActionTransformer(len(self.actions), int(cfg.get('d_model',128)), int(cfg.get('heads',4)), int(cfg.get('layers',2)), int(cfg.get('max_tokens',1024)), int(cfg.get('z_slots',8))).to(self.device)
                self.policy.load_state_dict(saved['state_dict']); self.policy.eval()
                print('Loaded recursive action policy:', checkpoint)
            except Exception as exc:
                print('Checkpoint unavailable; using random baseline:', exc)
                self.policy = None
    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        return latest_frame.state is GameState.WIN
    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):
            return GameAction.RESET
        if self.policy is None:
            action = self.rng.choice(self.actions)
            xy = (self.rng.randrange(64), self.rng.randrange(64))
        else:
            tokens = _observation_tokens(latest_frame, self.policy.max_tokens)
            x = torch.as_tensor(tokens.copy(), dtype=torch.long, device=self.device)[None]
            with torch.no_grad(): action_logits, xy_logits = self.policy(x)
            action = self.actions[int(action_logits[0].argmax().item())]
            xy = (int(xy_logits[0,0].argmax().item()), int(xy_logits[0,1].argmax().item()))
        if action.is_complex():
            action.set_data({'x': xy[0], 'y': xy[1]})
            action.reasoning = {'desired_action': str(action.value), 'my_reason': 'Recursive transformer policy' if self.policy is not None else 'Random baseline: no trained checkpoint'}
        elif action.is_simple():
            action.reasoning = 'Recursive transformer policy' if self.policy is not None else 'Random baseline: no trained checkpoint'
        return action

In [ ]:
import os, shutil
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Use the competition's required interactive agent runner.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 --retry-max-time 600 http://gateway:8001/api/games
    src = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents'
    dst = '/kaggle/working/ARC-AGI-3-Agents'
    shutil.copytree(src, dst, dirs_exist_ok=True)
    shutil.copy('/kaggle/working/my_agent.py', dst+'/agents/templates/my_agent.py')
    with open(dst+'/agents/__init__.py', 'w') as f:
        f.write('from typing import Type\nfrom dotenv import load_dotenv\nfrom .agent import Agent, Playback\nfrom .swarm import Swarm\nfrom .templates.random_agent import Random\nfrom .templates.my_agent import MyAgent\nload_dotenv()\nAVAILABLE_AGENTS: dict[str, Type[Agent]] = {\"random\": Random, \"myagent\": MyAgent}\n')
    with open(dst+'/.env', 'w') as f:
        f.write('SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\nARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE=online\nENVIRONMENTS_DIR=\nRECORDINGS_DIR=/kaggle/working/server_recording\n')
    !cd /kaggle/working/ARC-AGI-3-Agents && MPLBACKEND=agg python main.py --agent myagent
else:
    # Preserve the exact artifact format used by the supplied notebook scaffold.
    submission = pd.DataFrame([['1_0', '1', True, 1]], columns=['row_id','game_id','end_of_game','score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print('Created: /kaggle/working/my_agent.py and /kaggle/working/submission.parquet')
    print('submission.parquet columns:', list(submission.columns))